<a href="https://colab.research.google.com/github/BassemRamdan/AI-Resume-Intelligence/blob/main/EDA_and_Data_Splitting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Resume Intelligence - EDA and Data Splitting
This notebook downloads the resume dataset from Hugging Face, performs Exploratory Data Analysis (EDA), and splits the data into train, validation, and test sets.

In [ ]:
!pip install datasets huggingface_hub pandas matplotlib seaborn scikit-learn

In [ ]:
from datasets import load_dataset
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

# Load dataset from Hugging Face
print("Loading dataset from Hugging Face...")
dataset = load_dataset("BassemRamdan/data")
print(dataset)

In [ ]:
# Convert to pandas DataFrame for easier EDA
df = pd.DataFrame(dataset['train'])
print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())

# Basic information
print("\nDataset Info:")
df.info()

In [ ]:
# Visualize the distribution of resume categories
print("Columns in our dataset:", df.columns.tolist())

possible_label_cols = ['label', 'category', 'class', 'folder']
label_col = next((col for col in possible_label_cols if col in df.columns), None)

if not label_col and len(df.columns) > 1:
    label_col = df.columns[-1] # Fallback to last column

if label_col:
    plt.figure(figsize=(12, 8))
    sns.countplot(data=df, y=label_col, order=df[label_col].value_counts().index)
    plt.title('Distribution of Resume Categories')
    plt.xlabel('Count')
    plt.ylabel('Category')
    plt.tight_layout()
    plt.show()
else:
    print("Could not automatically find a label column for visualization. Please update the column name manually.")

In [ ]:
# Split the dataset into train, validation, and test sets (70% train, 15% val, 15% test)
print("Splitting data into Train, Validation, and Test sets...")

train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print(f"Train set size: {len(train_df)}")
print(f"Validation set size: {len(val_df)}")
print(f"Test set size: {len(test_df)}")

In [ ]:
# Optional: Convert back to HuggingFace Dataset and push the splits to Hub
from datasets import Dataset, DatasetDict

final_dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df.reset_index(drop=True)),
    'validation': Dataset.from_pandas(val_df.reset_index(drop=True)),
    'test': Dataset.from_pandas(test_df.reset_index(drop=True))
})

print(final_dataset)
# Uncomment and run to upload the splitted dataset back to Hugging Face
# final_dataset.push_to_hub("BassemRamdan/data-split")